# ChatML

Goal: executar os experimentos centrais do tutorial em Python.

Tutorial: `tutorials/01-chat-ml.md`


## Setup do ambiente

Este notebook implementa os exemplos do tutorial usando chamadas reais para a API legado de `completions` e o SDK oficial para o exemplo moderno.


In [ ]:
import os

import requests
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
legacy_model = "text-davinci-002"
client = OpenAI(api_key=api_key) if api_key else None

print("OPENAI_API_KEY carregada" if api_key else "Defina OPENAI_API_KEY no .env e inicie com npm run notebook ou npm run lab")
print("Modelo legado configurado:", legacy_model)


In [ ]:
def call_legacy_completion(prompt, max_tokens=120, temperature=0.7, stop=None):
    if not api_key:
        raise RuntimeError("Defina OPENAI_API_KEY no .env antes de executar.")

    payload = {
        "model": legacy_model,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    if stop is not None:
        payload["stop"] = stop

    response = requests.post(
        "https://api.openai.com/v1/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        },
        json=payload,
        timeout=60,
    )

    data = response.json()

    if not response.ok:
        print(data)
        raise RuntimeError(f"Legacy completions request failed: {response.status_code}")

    return data


## Parte 1: `completions` continua a sequencia

Corresponde ao experimento do tutorial que envia apenas um texto simples e observa a continuacao gerada.


In [ ]:
hello_response = call_legacy_completion(
    prompt="Ola",
    max_tokens=20,
    temperature=0.7,
)

print(hello_response["choices"][0]["text"])
print("Finish reason:", hello_response["choices"][0]["finish_reason"])


## Parte 2: formato de conversa via prompt estruturado

Aqui o objetivo e inspecionar como o modelo continua um prompt no formato `User:` / `Assistant:` sem controle de parada.


In [ ]:
prompt_as_dialogue = "User: Ola, tudo bem?\nAssistant:\n"

conversation_response = call_legacy_completion(
    prompt=prompt_as_dialogue,
    max_tokens=150,
    temperature=0.7,
)

print(conversation_response["choices"][0]["text"])


## Parte 3: `stop` controla o encerramento do turno

O codigo abaixo executa o mesmo prompt anterior, agora com corte explicito antes do proximo turno do usuario.


In [ ]:
stop_controlled_response = call_legacy_completion(
    prompt=prompt_as_dialogue,
    max_tokens=150,
    temperature=0.7,
    stop=["User:"],
)

print(stop_controlled_response["choices"][0]["text"])
print("Finish reason:", stop_controlled_response["choices"][0]["finish_reason"])


## Parte 4: ChatML explicita papeis e fronteiras

Este bloco serializa as mensagens manualmente em ChatML e usa `stop` para encerrar no fim do bloco do assistant.


In [ ]:
def to_chatml(messages):
    return "\n".join(
        f"<|im_start|>{message['role']}\n{message['content']}\n<|im_end|>"
        for message in messages
    )


chatml_prompt = (
    to_chatml([
        {"role": "system", "content": "Voce e um assistente util."},
        {"role": "user", "content": "Explique computacao em nuvem em 2 frases."},
    ])
    + "\n<|im_start|>assistant\n"
)

chatml_response = call_legacy_completion(
    prompt=chatml_prompt,
    max_tokens=120,
    temperature=0.7,
    stop=["<|im_end|>"],
)

print(chatml_response["choices"][0]["text"])
print("Finish reason:", chatml_response["choices"][0]["finish_reason"])


## Parte 5: equivalente moderno com SDK

O tutorial fecha conectando o formato legado a APIs modernas. Aqui usamos o SDK oficial da OpenAI para mostrar essa diferenca de interface.


In [ ]:
if not client:
    raise RuntimeError("Defina OPENAI_API_KEY no .env antes de executar.")

modern_response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": "Voce e um assistente util."},
        {"role": "user", "content": "Explique computacao em nuvem em 2 frases."},
    ],
)

print(modern_response.output_text)
